# Etna Dataset

This notebook builds the Etna case-study dataset for the Cause–Trigger analysis. Waveform features are extracted on an hourly grid, then merged with gas and meteorological context variables. The final dataset is used later for HMML/PCMCI-based trigger analysis.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src/etna"
sys.path.append(str(SRC_DIR))

from etna_config import (
    ETNA_WAVEFORM_CONFIG,
    ETNA_GAS_METEO_COLS,
    ETNA_EVENT_TIME,
    ETNA_WEATHER_COLS,
    etna_observable_metadata,
)

from etna_waveform import build_station_waveform_dataset

from etna_dataset import (
    load_etnagas_csv,
    extract_plume_co2so2_xls,
    load_etna_event_catalog_xls,
    create_etna_final_dataset,
    load_openmeteo_etna_weather,
)

from etna_plotting_utils import (
    dataset_health_report,
    plot_variable_pdfs,
    distribution_summary,
    run_teleseismic_checks,
    plot_etna_thesis_figures,
    plot_etna_all_variables_map,
    plot_etna_loglog_distributions,
    plot_etna_catalogue_counts,
)

from etna_stations import (
    run_station_screening,
    export_station_screening_results,
)

client = Client("INGV")
cfg = ETNA_WAVEFORM_CONFIG
ETNA_STATION = ETNA_WAVEFORM_CONFIG["stations"][0]

import warnings
from obspy.io.mseed import InternalMSEEDWarning

warnings.filterwarnings(
    "ignore",
    category=InternalMSEEDWarning,
    message=r".*fractional second.*10000.*",
)
warnings.filterwarnings(
    "ignore",
    message=r".*fractional seconds.*10000.*",
)


## Etna station availability screening for full analysis window

This section identifies the best Etna seismic station(s) for the full analysis window without downloading the full waveform archive. It first discovers all available vertical channels, then probes a short waveform segment on each day of the study period. Stations are ranked by daily availability and distance to the Etna summit.

Keep `RUN_STATION_SCREENING = False` for normal dataset construction. Set it to `True` only if you want to rerun the reproducible station-selection check and export fresh CSV/TXT reports.


In [ ]:
RUN_STATION_SCREENING = False  # Set to True to run station screening, False to skip and use selected station from config

if RUN_STATION_SCREENING:
    streams_found, daily_probe, station_summary = run_station_screening(
        client=client,
        waveform_cfg=ETNA_WAVEFORM_CONFIG,
        max_distance_km=80.0,
        probe_offset_hours=12,
        probe_duration_minutes=10,
        verbose=True,
        display_func=display,
    )

    export_paths = export_station_screening_results(
        station_summary=station_summary,
        daily_probe=daily_probe,
        out_dir="etna_stations",
        waveform_cfg=ETNA_WAVEFORM_CONFIG,
        probe_offset_hours=12,
        probe_duration_minutes=10,
    )

    print("\nRanked station-channel candidates:")
    display(station_summary)

    print("\nBest complete candidates:")
    display(station_summary.query("ok_fraction == 1.0"))

    print("\nSaved station-screening files:")
    for name, path in export_paths.items():
        print(f"{name}: {path}")
else:
    print(
        "Station screening skipped. "
        "Using selected station from ETNA_WAVEFORM_CONFIG: "
        f"{ETNA_STATION}.{ETNA_WAVEFORM_CONFIG['location']}.{ETNA_WAVEFORM_CONFIG['channel']}"
    )

## Selected station and channel metadata

We inspect the selected ESLN HHZ stream metadata. ESLN was selected because it is close to the Etna summit, has complete retrievable data over the full analysis window, and clearly records the Wenchuan wave arrival.


In [ ]:
inventory = client.get_stations(
    network=ETNA_WAVEFORM_CONFIG["network"],
    station=ETNA_STATION,
    location=ETNA_WAVEFORM_CONFIG["location"],
    channel=ETNA_WAVEFORM_CONFIG["channel"],
    level="response",
    starttime=ETNA_WAVEFORM_CONFIG["start"],
    endtime=ETNA_WAVEFORM_CONFIG["end"],
)

print(inventory)

## Hourly waveform feature extraction

Waveform data are processed in daily chunks with padding for filter stability.


In [ ]:
esln_wave, esln_failures = build_station_waveform_dataset(
    client=client,
    station=ETNA_STATION,
    cfg=cfg,
    cache_path="../data/etna/etna_esln_hhz_waveform.pkl",
    redownload=True, #set True to force redownloading, False to use cached dataset
)

print("failures:", esln_failures)
print("shape:", esln_wave.shape)
display(esln_wave.head())
display(esln_wave.tail())
display(esln_wave.isna().mean())

## External variables

These variables are merged onto the hourly waveform grid:

- WindSpeed [1] 
- CO2_3 [1] 
- pressure_drop (derived from Patm_3 column) [1]
- plume data (SO2/CO2 ratio) [2]
- Antecedent Precipitation Index (API) (derived from Open-Meteo rainfall data) [3]

[1] Liuzzo, M., Giuffrida, G. B., & Gurrieri, S. (2025). Etna CO2 Soil Flux during 2002-2010 (ECSF2002_2010) [Data set]. Istituto Nazionale di Geofisica e Vulcanologia (INGV). https://doi.org/10.13127/etna/ecsf2002_2010

[2] Aiuppa A., Giudice G., Liuzzo M. (2017). Volcanic gas plume data from Etna Volcano (Italy) (Version 1.0) [Data set]. Interdisciplinary Earth Data Alliance (IEDA). https://doi.org/10.1594/ieda/1006438

[3] Zippenfenig, P. (2023). Open-Meteo.com Weather API [Computer software]. Zenodo. https://doi.org/10.5281/ZENODO.7970649


In [ ]:
plume_df = extract_plume_co2so2_xls("../data/etna/1012-1_VolcanicGas_Etna.xls")

In [ ]:
etnagas_df = load_etnagas_csv(
    path="../data/etna/3c.csv",
    value_cols=ETNA_GAS_METEO_COLS,
)

display(etnagas_df.head())
display(etnagas_df.isna().mean().sort_values())

In [ ]:
etna_weather_df = load_openmeteo_etna_weather(
    start_date="2008-04-12",
    end_date="2008-05-16",
)

## Effect: Local event rate anomaly

Alparone, S. C., Maiolino, V., Mostaccio, A., Scaltrito, A., Ursino, A., Barberi, G., et al. (2015). Mt. Etna Seismic Catalog 2000-2010 [Data set]. Istituto Nazionale di Geofisica e Vulcanologia (INGV) - Osservatorio Etneo. https://doi.org/10.13127/etnasc/2000_2010

In [ ]:
etna_catalog = load_etna_event_catalog_xls(
    "../data/etna/Etna catalogue_2000-2010.xls",
    quality_filter=False,
)

In [ ]:
events_case = etna_catalog[
    (etna_catalog["timestamp"] >= EVENT_TIME - pd.Timedelta(days=3))
    & (etna_catalog["timestamp"] <= EVENT_TIME + pd.Timedelta(hours=36))
]

hourly_counts = (
    events_case
    .set_index("timestamp")
    .assign(count=1)
    ["count"]
    .resample("1h")
    .sum()
)

display(hourly_counts[hourly_counts > 0])

In [ ]:
plot_etna_catalogue_counts(

## Merge, scale, and save final hourly datasets

The waveform, gas, meteorological, and plume variables are merged by timestamp. Raw variables are retained, and scaled versions are produced for causal discovery. 

In [ ]:
etna_raw, etna_final = create_etna_final_dataset(
    wave_df=esln_wave,
    station_name=ETNA_STATION,
    catalog_df=etna_catalog,
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_GAS_METEO_COLS,
    plume_df=plume_df,
    weather_df=etna_weather_df,
    weather_cols=ETNA_WEATHER_COLS,
    output_dir="../data/etna"
)


## Dataset checks

We check dataset size, timestamp order, duplicate timestamps, missing values, and variable distributions. These checks ensure that the final hourly dataset is suitable for causal discovery.

In [ ]:
station_data = {
    ETNA_STATION: {
        "wave": esln_wave,
        "final_raw": etna_raw,
        "final_scaled": etna_final,
    },
}


In [ ]:
for sta, d in station_data.items():
    dataset_health_report(d["final_raw"], f"{sta} hourly raw")
    dataset_health_report(d["final_scaled"], f"{sta} hourly scaled")

## Distribution diagnostics

Empirical density plots and summary statistics are used to inspect skewness, outliers, and scaling behavior. 


In [ ]:
plot_variable_pdfs(etna_raw, filename="etna_pdf.png")

### distribution summary table

In [ ]:
summary_esln = distribution_summary(etna_raw)
display(summary_esln)


### Etna raw-variable log-log distributions

a visual diagnostic for scale, tails, and zero inflation.

In [ ]:
fig, axes, etna_loglog_report = plot_etna_loglog_distributions(
    etna_raw,
    save_dir="figures",
)

display(etna_loglog_report)

In [ ]:
model_cols = [c for c in etna_final.columns if c != "time"]
scaled_check = etna_final[model_cols].agg(["mean", "std", "min", "max"]).T
scaled_check["std_ddof0"] = etna_final[model_cols].std(ddof=0)

display(scaled_check)
display(etna_final[model_cols].abs().max().sort_values(ascending=False))

## Teleseismic arrival diagnostics

We inspect the waveform around the Wenchuan earthquake arrival. The 1-minute RMS is compared with hourly maximum aggregation to verify that the hourly grid preserves the arrival-period energy. The spectrogram confirms that the selected frequency bands capture the relevant trigger and response components.

In [ ]:
waveform_client = Client(
    base_url="https://webservices.ingv.it",
    service_mappings={
        "dataselect": "https://webservices.ingv.it/fdsnws/dataselect/1/",
        "station": "https://webservices.ingv.it/fdsnws/station/1/",
        "event": "https://webservices.ingv.it/fdsnws/event/1/",
    },
    _discover_services=False,
)

results = {}

results[ETNA_STATION] = run_teleseismic_checks(
    client=waveform_client,
    station=ETNA_STATION,
    cfg=ETNA_WAVEFORM_CONFIG,
    event_time=ETNA_EVENT_TIME,
)

## Final hourly dataset with event marker

The final dataset is plotted with the teleseismic event time marked. This provides a visual check that the causal-analysis dataset aligns with the known event window.

In [ ]:
plot_etna_thesis_figures(
    csv_path="../data/etna/etna_raw.csv",
    event_time=ETNA_EVENT_TIME,
    save_dir="figures",
    include_titles=False,
)

## Geographic Visualization

In [ ]:
etna_meta = etna_observable_metadata()

fig, ax, etna_variable_table = plot_etna_all_variables_map(
    metadata=etna_meta,
    satellite=False, # True for satellite imagery
    save_dir="figures",
    filename="etna_map",
)

etna_variable_table